# Timestamp resolution and Hawkes MLE: reproduce the paper's results
Runs the simulation study from the workshop paper. **QUICK mode** (default) uses 30 replications per cell and finishes in a few minutes; set `QUICK = False` for the paper's 200 replications (about 10 to 20 minutes on Colab).

Seeds are the same as in the paper's scripts, so the first replications match. Exact digits can differ slightly across NumPy, SciPy and Numba versions, and with 30 replications percentages are noisy. The qualitative pattern should reproduce: no collapse for small resolution, collapse around 0.3 to 0.4 (earlier for n=0.3, later for small n), dither stays accurate.

In [ ]:
!pip -q install numba scipy matplotlib pandas 2>&1 | tail -1

In [ ]:
"""Exponential-kernel Hawkes: simulation, likelihood, MLE, timestamp corruptions.
Intensity: lam(t) = mu + n*beta * sum_{t_i<t} exp(-beta (t - t_i)),  n = branching ratio.
Time unit is chosen so that the TRUE beta = 1; resolution delta is therefore in units of
the kernel decay time (delta*beta = delta)."""
import numpy as np
from numba import njit
from scipy.optimize import minimize

@njit(cache=True)
def simulate(mu, n, beta, T, seed):
    np.random.seed(seed)
    out = np.empty(2_000_000)
    k = 0
    t = 0.0
    S = 0.0            # sum_j n*beta*exp(-beta(t-t_j)) at current t
    while True:
        lam_bar = mu + S
        w = -np.log(np.random.random()) / lam_bar
        t_new = t + w
        if t_new > T:
            break
        S *= np.exp(-beta * w)
        t = t_new
        if np.random.random() * lam_bar <= mu + S:
            out[k] = t
            k += 1
            S += n * beta
    return out[:k].copy()

@njit(cache=True)
def nll(params, t, T):
    mu, n, beta = params[0], params[1], params[2]
    A = 0.0
    ll = 0.0
    comp = 0.0
    prev = t[0]
    for i in range(t.shape[0]):
        if i > 0:
            A = np.exp(-beta * (t[i] - prev)) * (1.0 + A)
        lam = mu + n * beta * A
        ll += np.log(lam)
        comp += 1.0 - np.exp(-beta * (T - t[i]))
        prev = t[i]
    return -(ll - mu * T - n * comp)

def fit(t, T, starts=((0.5, 0.3, 1.0), (0.3, 0.6, 3.0))):
    """MLE in (log mu, logit n, log beta). Returns (mu, n, beta, nll, hit_bound)."""
    BMAX = 1e4
    def f(z):
        mu = np.exp(z[0]); n = 1/(1+np.exp(-z[1])); b = np.exp(z[2])
        n = min(n, 0.9999)
        return nll(np.array([mu, n, b]), t, T)
    rate = len(t) / T
    best = None
    for (m0, n0, b0) in starts:
        z0 = np.array([np.log(max(rate*(1-n0), 1e-6)), np.log(n0/(1-n0)), np.log(b0)])
        r = minimize(f, z0, method="L-BFGS-B",
                     bounds=[(-12, 8), (-8, 9), (np.log(1e-3), np.log(BMAX))])
        if best is None or r.fun < best.fun:
            best = r
    mu = float(np.exp(best.x[0])); n = float(1/(1+np.exp(-best.x[1]))); b = float(np.exp(best.x[2]))
    return mu, n, b, float(best.fun), bool(b > 0.99 * BMAX)

# ---- timestamp corruptions -------------------------------------------------
def round_down(t, delta):
    return np.floor(t / delta) * delta

def dither(t_rounded, delta, rng):
    return np.sort(t_rounded + rng.random(len(t_rounded)) * delta)

def spread(t_rounded, delta):
    """Evenly spread k tied events across their bin: positions (j+0.5)/k * delta."""
    out = t_rounded.copy()
    u, idx, cnt = np.unique(t_rounded, return_index=True, return_counts=True)
    for s, c in zip(idx, cnt):
        if c > 1:
            out[s:s+c] = t_rounded[s] + (np.arange(c) + 0.5) / c * delta
    return out

def tie_fraction(t):
    return float(np.mean(np.diff(t) == 0)) if len(t) > 1 else 0.0

def add_noise(t, sigma, rng):
    return np.sort(t + rng.normal(0, sigma, len(t)))


In [ ]:
QUICK = True
REPS = 30 if QUICK else 200
MU, BETA, T = 1.0, 1.0, 3000.0
NS = [0.5, 0.8]
DELTAS = [0.03, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]
import time, pandas as pd
from multiprocessing import Pool
def one(args):
    n, rep = args
    seed = int(1000*n) * 100000 + rep          # same seeds as the paper's run_exp.py
    ts = simulate(MU, n, BETA, T, seed)
    rng = np.random.default_rng(seed + 7)
    rows = []
    o = fit(ts, T); rows.append(dict(n=n, rep=rep, level=0.0, method="oracle", tie=0.0, nh=o[1], b=o[2]))
    for d in DELTAS:
        tr = round_down(ts, d); tie = tie_fraction(tr)
        for name, x in (("raw", tr), ("dither", dither(tr, d, rng))):
            r = fit(x, T); rows.append(dict(n=n, rep=rep, level=d, method=name, tie=tie, nh=r[1], b=r[2]))
    return rows
t0 = time.time()
jobs = [(n, r) for n in NS for r in range(REPS)]
with Pool(2) as p: out = p.map(one, jobs, chunksize=2)
df = pd.DataFrame([r for rr in out for r in rr])
df["err"] = df.nh - df.n; df["collapsed"] = df.b > 100*BETA
print("done in %.0f s, %d fits" % (time.time()-t0, len(df)))

## Table: error in n-hat and collapse rate (compare with the paper's Table of rounded timestamps)

In [ ]:
g = df[df.method!="oracle"].groupby(["n","level","method"]).agg(ties=("tie","mean"), mean_err=("err","mean"), sd=("err","std"), collapse_pct=("collapsed", lambda x: 100*x.mean())).round(3)
print("oracle (unrounded) mean error / sd:"); print(df[df.method=="oracle"].groupby("n").err.agg(["mean","std"]).round(3))
g.reset_index().pivot_table(index=["n","level"], columns="method", values=["mean_err","collapse_pct"])

## Figures: collapse rate and error in n-hat vs resolution

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for n, c in zip(NS, ["tab:blue","tab:red"]):
    s = df[(df.n==n)&(df.method=="raw")].groupby("level").collapsed.mean()*100
    ax[0].plot(s.index, s.values, "o-", color=c, label=f"n={n}")
ax[0].set_xscale("log"); ax[0].set_xlabel("resolution  δβ"); ax[0].set_ylabel("% raw fits collapsed"); ax[0].legend(); ax[0].axvline(0.3, ls=":", c="gray")
for n, ls in zip(NS, ["-","--"]):
    for m, c in (("raw","tab:red"),("dither","tab:green")):
        s = df[(df.n==n)&(df.method==m)].groupby("level").err.mean()
        ax[1].plot(s.index, s.values, ls, marker="o", color=c, label=f"{m}, n={n}")
ax[1].axhline(0, c="k", lw=.6); ax[1].set_xscale("log"); ax[1].set_xlabel("resolution  δβ"); ax[1].set_ylabel("mean error in n-hat"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("collapse_and_bias.png", dpi=150); plt.show()

## Check of the theory: with any exact tie the likelihood is unbounded in β
Profile log-likelihood (maximized over μ, n) at β = 10, 100, 1000, 10000 for rounded data. It should keep rising by about k·ln(10) per tenfold increase of β, where k is the number of ties.

In [ ]:
ts = simulate(MU, 0.8, BETA, T, 80000)
tr = round_down(ts, 1.0); k = int(np.sum(np.diff(tr)==0)); print("events", len(tr), "ties k =", k, " k*ln10 =", round(k*np.log(10),1))
def profile(beta):
    f = lambda z: nll(np.array([np.exp(z[0]), min(1/(1+np.exp(-z[1])),0.9999), beta]), tr, T)
    r = minimize(f, np.array([np.log(0.3), 0.0]), method="L-BFGS-B", bounds=[(-12,8),(-8,9)])
    return -r.fun
prev=None
for b in [10,100,1000,10000]:
    v=profile(b); print(f"beta={b:>6}  profile loglik={v:12.1f}", "" if prev is None else f"  rise={v-prev:9.1f}"); prev=v

## Apply steps 1 to 4 of the checklist to your own timestamps
Set `path` to a CSV with one column of event times (any unit, sorted or not) and `resolution` to the recording resolution in the same unit. This has been tested on simulated data only.

In [ ]:
path = None          # e.g. "trades.csv"  (one column, no header)
resolution = None    # e.g. 0.001 for millisecond stamps in seconds
if path is not None:
    t = np.sort(pd.read_csv(path, header=None).iloc[:,0].to_numpy(float)); t = t - t[0]; T_ = t[-1] + resolution
    print("events", len(t), "tie fraction", round(tie_fraction(t), 3))
    raw = fit(t, T_); print("raw   : n-hat %.3f  beta-hat %.4g  at bound: %s" % (raw[1], raw[2], raw[4]))
    rng = np.random.default_rng(0)
    ests = []
    for _ in range(5):
        r = fit(np.sort(t + rng.random(len(t))*resolution), T_); ests.append((r[1], r[2]))
    ests = np.array(ests); print("dither: n-hat %.3f +- %.3f   beta-hat %.4g   delta*beta-hat = %.3f" % (ests[:,0].mean(), ests[:,0].std(), ests[:,1].mean(), resolution*ests[:,1].mean()))
else:
    print("set path and resolution to run")